In [3]:
# ==============================================================================
#                         STANDALONE ROBUST RUN ANALYZER
# ==============================================================================

import os
import json
import logging
from pathlib import Path
from typing import Optional, Callable, Dict, Any, List

import pandas as pd
from IPython.display import display

from statistical.statistical_framework import DataFrameStylerAuto
from themes import themes

# ==============================================================================
#                            CONFIGURATION PANEL
# ==============================================================================

STUDY_TO_VALIDATE        = "F14_FULL_RobustScaler_V01"
N_TOP_MODELS_TO_VALIDATE = 10

project_root             = os.path.abspath(os.path.join(os.getcwd(), '../..'))
RUN_NAME                 = f"validation_{STUDY_TO_VALIDATE}_top_{N_TOP_MODELS_TO_VALIDATE}"
EXPERIMENTS_DIR          = os.path.join(project_root, 'src', 'experiment_configs', 'results')
dir_path                 = os.path.join(project_root, "experiments", STUDY_TO_VALIDATE)
excel_path               = os.path.join(dir_path, "results.xlsx")

os.makedirs(dir_path, exist_ok=True)

In [6]:
def parse_single_json_result(file_path: Path) -> List[Dict]:
    """
    Parses a single result JSON file and extracts all metric rows,
    correctly handling nested lists for slice metrics.
    """
    with open(file_path, 'r') as f:
        data = json.load(f)

    if data.get("status") != "success":
        return []

    base_info = {
        'experiment_id': data.get('experiment_id'),
        'well': data.get('well'),
        'strategy': data.get('config', {}).get('strategy_config', {}).get('strategy_name'),
        'extractor': data.get('config', {}).get('extractor_config', {}).get('type'),
        'fuser': data.get('config', {}).get('fuser_config', {}).get('type'),
    }

    parsed_rows = []
    job_results = data.get("results", {})

    for key, metric_data in job_results.items():
        # We only care about keys that represent metric results
        if not key.endswith(('_val', '_test')):
            continue

        if not metric_data:
            continue

        # --- THE DEFINITIVE FIX for nested lists ---
        
        # Normalize the data into a flat list of dictionaries
        rows_to_process = []
        if isinstance(metric_data, dict):
            # Handles global_metrics (single dict)
            rows_to_process = [metric_data]
        elif isinstance(metric_data, list):
            # Check if it's a list of lists (for slices)
            if all(isinstance(item, list) for item in metric_data):
                # Flatten the list of lists into a single list of dicts
                rows_to_process = [d for sublist in metric_data for d in sublist]
            else:
                # It's a regular list of dicts (agg/cum metrics)
                rows_to_process = metric_data

        # Now, process the normalized list of rows
        for row_dict in rows_to_process:
            if isinstance(row_dict, dict): # Final safety check
                full_row = {**row_dict, **base_info}
                full_row['metric_source'] = key
                parsed_rows.append(full_row)
                
    return parsed_rows

def create_legacy_report_from_jsons(run_dir: Path, excel_path: Path) -> None:
    """
    Main orchestrator: loads all JSONs, transforms the data, displays a summary,
    and saves a legacy-compatible results.xlsx file.
    """
    results_dir = run_dir / "results"
    if not results_dir.is_dir():
        print(f"❌ Error: Results directory not found at {results_dir}")
        return

    # 1. Load and parse all JSONs into one big list of records
    all_records = []
    json_files = sorted(results_dir.glob("*.json"))
    if not json_files:
        print("No JSON result files found.")
        return
        
    for json_file in json_files:
        all_records.extend(parse_single_json_result(json_file))

    if not all_records:
        print("No successful jobs with valid metrics found.")
        return
        
    master_df = pd.DataFrame(all_records)
    
    # 2. Separate Aggregated and Cumulative metrics for the final report
    agg_df = master_df[master_df['Category'].str.contains('Aggregated', na=False)].copy()
    cum_df = master_df[master_df['Category'].str.contains('Cumulative', na=False)].copy()

    # 3. Pivot the data to get VAL and TEST side-by-side
    def pivot_metrics(df: pd.DataFrame, category_name: str) -> pd.DataFrame:
        if df.empty:
            return pd.DataFrame()
        
        # Separate val and test data
        val = df[df['metric_source'].str.contains('_val')].copy()
        test = df[df['metric_source'].str.contains('_test')].copy()
        
        # Rename metric columns
        val = val.rename(columns={'SMAPE': 'SMAPE_VAL', 'MAE': 'MAE_VAL'})
        test = test.rename(columns={'SMAPE': 'SMAPE_TEST', 'MAE': 'MAE_TEST'})
        
        # The key to identify a unique row
        id_vars = ['experiment_id', 'well', 'strategy', 'extractor', 'fuser']
        
        # Merge them. Since each JSON is one experiment, this merge should be clean.
        merged_df = pd.merge(
            val[id_vars + ['SMAPE_VAL', 'MAE_VAL']],
            test[id_vars + ['SMAPE_TEST', 'MAE_TEST']],
            on=id_vars,
            how='outer'
        )
        return merged_df

    agg_report_df = pivot_metrics(agg_df, "Aggregated").drop_duplicates()
    cum_report_df = pivot_metrics(cum_df, "Cumulative").drop_duplicates()

    # Defina a ordem e as colunas desejadas
    final_cols = [
        'well', 'strategy', 'extractor', 'fuser',
        'MAE_VAL', 'MAE_TEST', 'SMAPE_VAL', 'SMAPE_TEST'
    ]
    
    # Reordene e remova 'experiment_id'
    agg_report_df = agg_report_df[final_cols]
    cum_report_df = cum_report_df[final_cols]


    # 4. Display the styled summary
    print(f"\n--- Analysis for Run: {run_dir.name} ---")
    
    print("\n--- Aggregated Metrics Summary ---")
    if not agg_report_df.empty:
        styled_agg = DataFrameStylerAuto.style_dataframe(
            agg_report_df.sort_values("SMAPE_VAL"), "minimal", themes=themes
        )
        display(styled_agg)
    
    print("\n--- Cumulative Metrics Summary ---")
    if not cum_report_df.empty:
        styled_cum = DataFrameStylerAuto.style_dataframe(
            cum_report_df.sort_values("SMAPE_VAL"), "minimal", themes=themes
        )
        display(styled_cum)
        
    # 5. Save the results.xlsx file
    with pd.ExcelWriter(excel_path) as writer:
        agg_report_df.to_excel(writer, sheet_name="aggregated_metrics", index=False)
        cum_report_df.to_excel(writer, sheet_name="cumulative_metrics", index=False)
        
    print(f"\n✅ Successfully created legacy-compatible 'results.xlsx' at: {excel_path}")

# ==============================================================================
#                                  EXECUTION
# ==============================================================================

# --- Configuration Panel ---
STUDY_TO_VALIDATE        = "F14_FULL_RobustScaler_V01"
N_TOP_MODELS_TO_VALIDATE = 10
RUN_NAME                = f"validation_{STUDY_TO_VALIDATE}_top_{N_TOP_MODELS_TO_VALIDATE}"
EXPERIMENTS_DIR         = os.path.join(project_root, 'src', 'experiment_configs', 'results')

dir_path                = os.path.join(project_root, "experiments", STUDY_TO_VALIDATE)
excel_path              = os.path.join(dir_path, "results.xlsx")

# --- Run the analysis ---
create_legacy_report_from_jsons(Path(EXPERIMENTS_DIR) / RUN_NAME, excel_path)


--- Analysis for Run: validation_F14_FULL_RobustScaler_V01_top_10 ---

--- Aggregated Metrics Summary ---
DataFrame shape: 10 rows x 8 columns; Index range: 0 to 36


,well,strategy,extractor,fuser,MAE_VAL,MAE_TEST,SMAPE_VAL,SMAPE_TEST
4,15/9-F-12,pressure_ensemble,tcn,bias_scale,1256.56,474.83,22.84,20.86
32,15/9-F-12,combined_exp_arps,identity,film,1271.95,806.35,23.44,34.78
36,15/9-F-12,pressure_ensemble,tcn,bias_scale,1365.18,545.28,23.84,23.64
16,15/9-F-12,exponential,rnn,bias_scale,1369.13,689.33,24.04,29.08
24,15/9-F-12,arps,rnn,bias_scale,1379.83,803.87,24.20,33.15
20,15/9-F-12,exponential,rnn,bias_scale,1357.16,750.02,24.39,32.14
0,15/9-F-12,exponential,cnn,bias_scale,1413.47,693.46,25.04,30.83
8,15/9-F-12,arps,aggregate,bias_scale,1423.68,517.75,25.61,23.06
28,15/9-F-12,exponential,rnn,bias_scale,1467.99,683.26,25.71,28.89
12,15/9-F-12,pressure_ensemble,aggregate,bias_scale,1512.61,505.13,27.91,23.37



--- Cumulative Metrics Summary ---
DataFrame shape: 10 rows x 8 columns; Index range: 0 to 36


,well,strategy,extractor,fuser,MAE_VAL,MAE_TEST,SMAPE_VAL,SMAPE_TEST
8,15/9-F-12,arps,aggregate,bias_scale,18056.69,53693.28,0.07,0.21
20,15/9-F-12,exponential,rnn,bias_scale,20900.61,249111.94,0.09,0.96
4,15/9-F-12,pressure_ensemble,tcn,bias_scale,40923.59,71023.60,0.17,0.27
16,15/9-F-12,exponential,rnn,bias_scale,44842.51,250872.23,0.18,0.97
28,15/9-F-12,exponential,rnn,bias_scale,47907.25,234863.66,0.20,0.91
24,15/9-F-12,arps,rnn,bias_scale,52743.14,334046.82,0.21,1.29
0,15/9-F-12,exponential,cnn,bias_scale,58985.88,219538.61,0.24,0.85
36,15/9-F-12,pressure_ensemble,tcn,bias_scale,73790.14,190371.03,0.30,0.73
32,15/9-F-12,combined_exp_arps,identity,film,82191.94,324323.41,0.34,1.24
12,15/9-F-12,pressure_ensemble,aggregate,bias_scale,88566.95,142652.84,0.36,0.55



✅ Successfully created legacy-compatible 'results.xlsx' at: /home/gabriel/Documentos/Equinor/experiments/F14_FULL_RobustScaler_V01/results.xlsx
